# Comparacion de intensidad: original vs enmascarada

Este notebook compara intensidades **solo dentro del ROI de la mascara** y guarda una imagen enmascarada.
Tambien incluye checks para detectar causas comunes de cambio de intensidad al aplicar mascara.


In [3]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import nibabel as nib
import nibabel.processing as nibproc
import matplotlib.pyplot as plt


## 1) Configura rutas

- `ORIGINAL_PATH`: imagen original (3D o 4D)
- `MASK_PATH`: mascara (3D o 4D)
- `MASKED_INPUT_PATH`: opcional, imagen ya enmascarada para comparar
- `OUTPUT_MASKED_PATH`: salida de imagen enmascarada generada aqui


In [7]:
ORIGINAL_PATH = Path("../output/Images_Congress/images/7T/input_mag_raw.nii.gz")
MASK_PATH = Path("../output/Images_Congress/images/cow_seg_final.nii.gz")
MASKED_INPUT_PATH = Path('../output/Images_Congress/images/7T_masked/input_mag_raw.nii.gz')
OUTPUT_MASKED_PATH = Path("../output/mask_qc/001_20240313_7T/input_mag_raw_masked.nii.gz")

MASK_THRESHOLD = 0.5
MASK_TIME_INDEX = 0  # si mascara es 4D y original es 3D
ATOL = 1e-6


In [8]:
def load_nifti_float(path: Path):
    img = nib.load(str(path))
    data = np.asarray(img.dataobj, dtype=np.float32)
    return img, data


def select_3d_mask(mask_data: np.ndarray, time_index: int = 0) -> np.ndarray:
    if mask_data.ndim == 3:
        return mask_data
    if mask_data.ndim == 4:
        if not (0 <= time_index < mask_data.shape[-1]):
            raise ValueError(f"time_index={time_index} fuera de rango para mascara shape={mask_data.shape}")
        return np.asarray(mask_data[..., time_index], dtype=np.float32)
    raise ValueError(f"Mascara invalida con shape={mask_data.shape}. Se esperaba 3D o 4D.")


def align_mask_to_reference(mask_img: nib.Nifti1Image, mask_3d: np.ndarray, ref_img: nib.Nifti1Image) -> np.ndarray:
    mask_3d_img = nib.Nifti1Image(mask_3d, mask_img.affine, mask_img.header)
    same_shape = tuple(mask_3d_img.shape[:3]) == tuple(ref_img.shape[:3])
    same_affine = np.allclose(mask_3d_img.affine, ref_img.affine, atol=1e-4)
    if not (same_shape and same_affine):
        mask_3d_img = nibproc.resample_from_to(mask_3d_img, (ref_img.shape[:3], ref_img.affine), order=0)
    return np.asarray(mask_3d_img.dataobj, dtype=np.float32)


def to_binary_mask(mask_data: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    return (mask_data >= float(threshold)).astype(np.float32)


def broadcast_mask(mask_3d: np.ndarray, target_shape: tuple[int, ...]) -> np.ndarray:
    if len(target_shape) == 3:
        return mask_3d
    if len(target_shape) == 4:
        return np.broadcast_to(mask_3d[..., np.newaxis], target_shape).astype(np.float32)
    raise ValueError(f"Shape no soportado: {target_shape}.")


def apply_mask(data: np.ndarray, mask_3d_binary: np.ndarray) -> np.ndarray:
    mask = broadcast_mask(mask_3d_binary, data.shape)
    return (data * mask).astype(np.float32)


def save_float32_nifti(data: np.ndarray, ref_img: nib.Nifti1Image, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    header = ref_img.header.copy()
    header.set_data_dtype(np.float32)
    header.set_slope_inter(1.0, 0.0)
    out_img = nib.Nifti1Image(data.astype(np.float32, copy=False), ref_img.affine, header)
    nib.save(out_img, str(out_path))


In [9]:
orig_img, orig = load_nifti_float(ORIGINAL_PATH)
mask_img, mask_raw = load_nifti_float(MASK_PATH)

mask_3d = select_3d_mask(mask_raw, time_index=MASK_TIME_INDEX)
mask_3d = align_mask_to_reference(mask_img, mask_3d, orig_img)
mask_bin = to_binary_mask(mask_3d, threshold=MASK_THRESHOLD)

masked_generated = apply_mask(orig, mask_bin)

masked_external = None
if MASKED_INPUT_PATH is not None:
    _, masked_external = load_nifti_float(Path(MASKED_INPUT_PATH))

print("Original shape:", orig.shape)
print("Mask shape (3D):", mask_bin.shape)
print("Mask voxels > 0:", int(mask_bin.sum()))
print("Original dtype(dataobj->float32):", orig.dtype)
print("Header dtype original:", orig_img.header.get_data_dtype())
print("Header slope/inter original:", orig_img.header.get_slope_inter())


Original shape: (147, 99, 48, 14)
Mask shape (3D): (147, 99, 48)
Mask voxels > 0: 15577
Original dtype(dataobj->float32): float32
Header dtype original: float32
Header slope/inter original: (None, None)


In [14]:
masked_external

array([[[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]],

        ...,

        [[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
    

In [10]:
mask_b = broadcast_mask(mask_bin, orig.shape) > 0
orig_in = orig[mask_b]
gen_in = masked_generated[mask_b]
diff_gen = gen_in - orig_in

print("Comparacion dentro de mascara (original vs enmascarada generada):")
print("  mean|diff|:", float(np.mean(np.abs(diff_gen))))
print("  max|diff| :", float(np.max(np.abs(diff_gen))))
print("  allclose   :", bool(np.allclose(orig_in, gen_in, atol=ATOL)))

if masked_external is not None:
    if masked_external.shape != orig.shape:
        raise ValueError(f"La imagen enmascarada externa tiene shape {masked_external.shape} y original {orig.shape}.")
    ext_in = masked_external[mask_b]
    diff_ext = ext_in - orig_in
    print("\nComparacion dentro de mascara (original vs enmascarada externa):")
    print("  mean|diff|:", float(np.mean(np.abs(diff_ext))))
    print("  max|diff| :", float(np.max(np.abs(diff_ext))))
    print("  allclose   :", bool(np.allclose(orig_in, ext_in, atol=ATOL)))


Comparacion dentro de mascara (original vs enmascarada generada):
  mean|diff|: 0.0
  max|diff| : 0.0
  allclose   : True

Comparacion dentro de mascara (original vs enmascarada externa):
  mean|diff|: 6.8639655113220215
  max|diff| : 277.4884948730469
  allclose   : False


In [15]:
import numpy as np, nibabel as nib

o = nib.load(str(ORIGINAL_PATH))
e = nib.load(str(MASKED_INPUT_PATH))

print("shape:", o.shape, e.shape)
print("affine_equal:", np.allclose(o.affine, e.affine, atol=1e-4))

def rep(img, name):
    h = img.header
    print(f"\n{name}")
    print("dtype header:", h.get_data_dtype())
    print("slope/inter:", h.get_slope_inter())
    raw = img.dataobj.get_unscaled() if hasattr(img.dataobj, "get_unscaled") else np.asarray(img.dataobj)
    print("raw dtype:", raw.dtype, "raw min/max:", float(np.min(raw)), float(np.max(raw)))
    dat = np.asarray(img.dataobj, dtype=np.float32)
    print("scaled min/max:", float(np.min(dat)), float(np.max(dat)))

rep(o, "ORIGINAL")
rep(e, "EXTERNA")

x = orig_in.astype(np.float64); y = ext_in.astype(np.float64)
a, b = np.polyfit(x, y, 1); corr = np.corrcoef(x, y)[0, 1]
print("\nfit y=a*x+b -> a:", a, "b:", b, "corr:", corr)


shape: (147, 99, 48, 14) (147, 99, 48, 14)
affine_equal: True

ORIGINAL
dtype header: float32
slope/inter: (None, None)
raw dtype: float32 raw min/max: -24.631589889526367 1668.09033203125
scaled min/max: -24.631589889526367 1668.09033203125

EXTERNA
dtype header: float32
slope/inter: (None, None)
raw dtype: float32 raw min/max: -15.278399467468262 1658.012451171875
scaled min/max: -15.278399467468262 1658.012451171875

fit y=a*x+b -> a: 0.9972193425255911 b: 1.7229398394814248 corr: 0.999153903534136


## 2) Relative scale dentro de la mascara\n
\n
Aqui se ajusta una recta entre voxel-a-voxel dentro del ROI:\n
\n
- `y = a*x + b`\n
- `x`: intensidad original\n
- `y`: intensidad enmascarada externa\n
\n
Interpretacion:\n
- `a` (scale): 1.0 ideal. Si `a < 1` hay compresion de escala; si `a > 1` hay amplificacion.\n
- `b` (offset): 0.0 ideal. Si es distinto de 0, hay desplazamiento aditivo.\n
- `corr`: cercania de la relacion lineal (1.0 ideal).\n

In [12]:
if masked_external is None:
    print("Define MASKED_INPUT_PATH para estimar relative scale contra la imagen enmascarada externa.")
else:
    x = orig_in.astype(np.float64)
    y = ext_in.astype(np.float64)

    a, b = np.polyfit(x, y, 1)
    corr = np.corrcoef(x, y)[0, 1]

    print("Ajuste lineal dentro de mascara: y = a*x + b")
    print("  scale (a):", float(a))
    print("  offset (b):", float(b))
    print("  corr      :", float(corr))

    eps_scale = 0.01
    eps_offset = 1e-3
    if abs(a - 1.0) <= eps_scale and abs(b) <= eps_offset:
        print("Interpretacion: no hay evidencia de cambio de escala relativa en ROI.")
    elif abs(a - 1.0) > eps_scale and abs(b) <= eps_offset:
        print("Interpretacion: hay cambio multiplicativo de escala (relative scale) en ROI.")
    elif abs(a - 1.0) <= eps_scale and abs(b) > eps_offset:
        print("Interpretacion: hay offset aditivo, pero no cambio relevante de escala.")
    else:
        print("Interpretacion: hay cambio de escala y offset en ROI.")

Ajuste lineal dentro de mascara: y = a*x + b
  scale (a): 0.9972193425255911
  offset (b): 1.7229398394814248
  corr      : 0.999153903534136
Interpretacion: hay offset aditivo, pero no cambio relevante de escala.


In [17]:
import nibabel as nib, numpy as np
from pathlib import Path

old_path = Path("../output/mask_qc/old_save.nii.gz")
new_path = Path("../output/mask_qc/new_save.nii.gz")

# masked_generated ya lo tienes del notebook
# orig_img también

# old save
nib.save(nib.Nifti1Image(masked_generated.astype(np.float32), orig_img.affine, orig_img.header), str(old_path))

# new save
h = orig_img.header.copy()
h.set_data_dtype(np.float32)
h.set_slope_inter(1.0, 0.0)
nib.save(nib.Nifti1Image(masked_generated.astype(np.float32), orig_img.affine, h), str(new_path))

old = np.asarray(nib.load(str(old_path)).dataobj, dtype=np.float32)
new = np.asarray(nib.load(str(new_path)).dataobj, dtype=np.float32)

m = (broadcast_mask(mask_bin, old.shape) > 0)
print("mean|old-new| in mask:", float(np.mean(np.abs(old[m]-new[m]))))
print("max|old-new| in mask :", float(np.max(np.abs(old[m]-new[m]))))


mean|old-new| in mask: 0.0
max|old-new| in mask : 0.0


In [16]:
save_float32_nifti(masked_generated, orig_img, OUTPUT_MASKED_PATH)
print(f"Imagen enmascarada guardada en: {OUTPUT_MASKED_PATH}")


Imagen enmascarada guardada en: ../output/mask_qc/001_20240313_7T/input_mag_raw_masked.nii.gz


## Posibles causas de cambio de intensidad (en el codigo de enmascarado)

1. Guardar con `header` original en dtype entero (`int16/uint16`) puede cuantizar/truncar valores.
2. Mantener `scl_slope/scl_inter` del header puede introducir escalado inesperado al re-leer.
3. Si la mascara no esta alineada a la imagen y se remuestrea mal, el ROI efectivo cambia.
4. Si se usa interpolacion no-nearest para mascara, pueden aparecer valores intermedios antes del umbral.

En este notebook se evita (1) y (2) guardando en `float32` con `slope=1` e `intercept=0`, y se controla (3)/(4) con alineacion + remuestreo `order=0`.
